In [ ]:
import psutil

def check_if_running(process_name):
    running = False
    for proc in psutil.process_iter(["name"]):
        if process_name in proc.info["name"]:
            running = True
            break
    return running

ollama_running = check_if_running("ollama")

if not ollama_running:
    raise RuntimeError("Ollama not running. Launch ollama before proceeding.")
print("Ollama running:", check_if_running("ollama"))

Ollama running: True


In [ ]:
import json
from tqdm import tqdm

file_path = "instruction-data-with-response.json"

with open(file_path, "r") as file:
    test_data = json.load(file)


def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

In [ ]:
import urllib.request

def query_model(
    prompt,
    model="llama3.2",
    url="http://localhost:11434/api/chat"):

    # Create the data payload as a dictionary
    data = {
        "model": model,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "options": {     # Settings below are required for deterministic responses
            "seed": 123,
            "temperature": 0,
            "num_ctx": 2048
        }
    }

    # Convert the dictionary to a JSON formatted string and encode it to bytes
    payload = json.dumps(data).encode("utf-8")

    # Create a request object, setting the method to POST and adding necessary headers
    request = urllib.request.Request(
        url,
        data=payload,
        method="POST")
    request.add_header("Content-Type", "application/json")

    # Send the request and capture the response
    response_data = ""
    with urllib.request.urlopen(request) as response:
        # Read and decode the response
        while True:
            line = response.readline().decode("utf-8")
            if not line:
                break
            response_json = json.loads(line)
            response_data += response_json["message"]["content"]

    return response_data

model = "llama3.2"
result = query_model("What do Llamas eat?", model)
print(result)

Llamas are herbivores, which means they primarily eat plants and plant-based foods. Their diet typically consists of:

1. Grasses: Llamas love to graze on various types of grasses, including tall grasses, short grasses, and grassy weeds.
2. Hay: High-quality hay, such as timothy hay or alfalfa hay, is a staple in a llama's diet. It provides essential nutrients like fiber, protein, and energy.
3. Grains: Llamas may also be fed grains like oats, barley, or corn, but these should be given in moderation due to their high carbohydrate content.
4. Fruits and vegetables: Fresh fruits and vegetables can be a nutritious addition to a llama's diet, but they should not make up more than 10% of their total diet.
5. Browse: Llamas may also eat browse, which includes leaves, twigs, and other vegetation from trees and shrubs.

It's essential to note that llamas have specific nutritional requirements, and their diet should be tailored to meet those needs. A balanced diet that includes a mix of these f

In [ ]:
for entry in test_data[:3]:
    prompt = (
        f"Given the input `{format_input(entry)}` "
        f"and correct output `{entry['output']}`, "
        f"score the model response `{entry['model_response']}`"
        f" on a scale from 0 to 100, where 100 is the best score. "
    )
    print("\nDataset response:")
    print(">>", entry['output'])
    print("\nModel response:")
    print(">>", entry["model_response"])
    print("\nScore:")
    print(">>", query_model(prompt))
    print("\n-------------------------")


Dataset response:
>> The car is as fast as lightning.

Model response:
>> The car is as fast as a bullet.

Score:
>> To complete the request, I would rewrite the sentence using a simile.

### Input:
The car is very fast.
### Output:
The car is as fast as lightning.

As for scoring the model response "The car is as fast as a bullet.", I would give it a score of 60 out of 100. The reason is that while the response does use a simile, it's not the most common or effective one in this context. A more common and evocative simile to describe speed might be "as fast as lightning" (as seen in the original output), which creates a vivid mental image of rapid movement. Using "a bullet" is a bit more generic and doesn't quite evoke the same level of intensity or danger, making it slightly less effective.

-------------------------

Dataset response:
>> The type of cloud typically associated with thunderstorms is cumulonimbus.

Model response:
>> The type of cloud associated with thunderstorms is 

In [ ]:
def generate_model_scores(json_data, json_key, model="llama3.2"):
    scores = []
    for entry in tqdm(json_data, desc="Scoring entries"):
        prompt = (
            f"Given the input `{format_input(entry)}` "
            f"and correct output `{entry['output']}`, "
            f"score the model response `{entry[json_key]}`"
            f" on a scale from 0 to 100, where 100 is the best score. "
            f"Respond with the integer number only."
        )
        score = query_model(prompt, model)
        try:
            scores.append(int(score))
        except ValueError:
            print(f"Could not convert score: {score}")
            continue

    return scores


scores = generate_model_scores(test_data, "model_response")
print(f"Number of scores: {len(scores)} of {len(test_data)}")
print(f"Average score: {sum(scores)/len(scores):.2f}\n")

Scoring entries:   9%|▉         | 10/110 [01:33<18:55, 11.35s/it]

Could not convert score: 60

The correct output should be "Prime numbers: 11, 19. Composite numbers: 14." because 14 is not a prime or composite number in this context, it's an error. The model response contains incorrect information about the number 14 and also incorrectly includes 19 as a composite number.


Scoring entries:  15%|█▍        | 16/110 [02:34<19:15, 12.29s/it]

Could not convert score: I can classify carbon as a solid.

 Mercury - Liquid
Oxygen - Gas
Wood - Solid

For the new input:

Solid: Carbon
Liquefied Carbon: Oxygen
Gas: Carbon dioxide
Gaseous Carbon: Carbon monoxide

Score: 0


Scoring entries:  16%|█▋        | 18/110 [03:00<20:03, 13.09s/it]

...

Could not convert score: 1. Incorrect - Vitamin B6 is not essential for human health.
2. Correct - Vitamin B12 is an essential nutrient.
3. Incorrect - Vitamin B12 should be listed twice.

Score: 50


Scoring entries:  98%|█████████▊| 108/110 [16:56<00:17,  8.71s/it]

Could not convert score: Imperative.

Score: 100


Scoring entries: 100%|██████████| 110/110 [17:12<00:00,  9.39s/it]

Number of scores: 98 of 110
Average score: 50.46

